In [ ]:
from pathlib import Path
import os, sys, datetime, subprocess

PACKAGE = Path.cwd()
if not (PACKAGE / "run.py").is_file():
    matches = list(PACKAGE.glob("**/progress3/run.py"))
    if Path("/kaggle/input").exists():
        matches += list(Path("/kaggle/input").glob("**/progress3/run.py"))
    if not matches:
        raise FileNotFoundError("Set PACKAGE to the extracted progress3 directory.")
    PACKAGE = matches[0].parent

DATA = PACKAGE / "data/comvis_dataset_v3"
if not (DATA / "metadata.csv").is_file():
    candidates = list(Path("/kaggle/input").glob("**/comvis_dataset_v3/metadata.csv")) if Path("/kaggle/input").exists() else []
    if not candidates:
        raise FileNotFoundError("Set DATA to the directory containing metadata.csv and splits/.")
    DATA = candidates[0].parent

BASE_OUTPUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else PACKAGE / "runs"
OUTPUT = BASE_OUTPUT / ("progress3_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))
DEVICE = "auto"
EPOCHS = 20
PATIENCE = 5
BATCH_SIZE = 32
AUGMENTATION_COPIES = 3
GLCM_DISTANCES = [1, 2, 4]
IMPORTANCE_REPEATS = 20
print("PACKAGE:", PACKAGE, "\nDATA:", DATA, "\nOUTPUT:", OUTPUT)


In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA/ROCm:", torch.cuda.is_available())
print("MPS:", torch.backends.mps.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
command = [
    sys.executable, str(PACKAGE / "run.py"), "all",
    "--data", str(DATA), "--output", str(OUTPUT), "--device", DEVICE,
    "--epochs", str(EPOCHS), "--patience", str(PATIENCE), "--batch-size", str(BATCH_SIZE),
    "--augmentation-copies", str(AUGMENTATION_COPIES),
    "--glcm-distances", *map(str, GLCM_DISTANCES),
    "--importance-repeats", str(IMPORTANCE_REPEATS),
]
print(" ".join(command))
subprocess.run(command, cwd=PACKAGE, check=True)


In [ ]:
import pandas as pd
from IPython.display import display, Image

display(pd.read_csv(OUTPUT / "comparison.csv"))
display(Image(filename=str(OUTPUT / "comparison.png")))
display(Image(filename=str(OUTPUT / "selected_dl_learning_curves.png")))

for name in ["svm_group_importance.png", "xgboost_group_importance.png", "dl_raw_channel_importance.png", "dl_pcd_channel_importance.png"]:
    display(Image(filename=str(OUTPUT / "importance" / name)))


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    subprocess.run([sys.executable, str(PACKAGE / "run.py"), "lock", "--data", str(DATA), "--output", str(OUTPUT)], cwd=PACKAGE, check=True)
    subprocess.run([sys.executable, str(PACKAGE / "run.py"), "test", "--data", str(DATA), "--output", str(OUTPUT), "--device", DEVICE], cwd=PACKAGE, check=True)
    display(pd.read_csv(OUTPUT / "comparison.csv"))


In [ ]:
SAMPLE_IMAGE = None
if SAMPLE_IMAGE is not None:
    subprocess.run([sys.executable, str(PACKAGE / "predict.py"), "--image", str(SAMPLE_IMAGE), "--results", str(OUTPUT), "--device", DEVICE], cwd=PACKAGE, check=True)
